# HBV hydrological model forced with DestinE historical forcing data
In this notebook we generate forcing data for the HBV hydrological model from the DestinE Climate DT CMIP6 historical simulation, using the ICON model (1990–~2020).

Unlike the SSP3-7.0 future data (step_1c) which is available directly as zarr on Cache B, the historical simulation is not yet available as a zarr store. It is accessed via the **polytope API** using `earthkit.data`. The data is returned in GRIB format on a HEALPix grid and is regridded to a regular lat/lon grid before being clipped to the catchment shape.

Authentication uses the same DESP credentials as step_1c. Make sure `DESP_USERNAME` and `DESP_PASSWORD` are set in your `.env` file.

For detailed descriptions of the forcing generation steps, see [step_1a](step_1a_generate_historical_forcing.ipynb).

In [ ]:
# General python
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import json
import sys

# Niceties
from rich import print

# This will work normally for HPC
try:
    from scripts.forcing_destine import DestinEHistoricalForcing
    from scripts.dest_auth import authenticate as dest_auth
except ImportError:
    # Add the project root to Python path for use on SRC
    project_root = Path().resolve().parent
    sys.path.append(str(project_root))

    from scripts.forcing_destine import DestinEHistoricalForcing
    from scripts.dest_auth import authenticate as dest_auth

dest_auth()

In [ ]:
# General eWaterCycle
import ewatercycle
import ewatercycle.forcing

In [ ]:
# Parameters, these get changed when running on HPC
country = "australia"
region_id = "camelsaus_102101A"
settings_path = "settings.json"

In [ ]:
# Load settings
with open(settings_path, "r") as json_file:
    settings = json.load(json_file)

In [ ]:
display(settings)

## DestinE CMIP6 historical forcing

The DestinE Climate DT provides a CMIP6 historical simulation using the ICON model at ~40 km resolution (HEALPix H128 grid, referred to as "standard" resolution). The historical run covers 1990 to ~2020, overlapping with both the calibration (1994–2004) and validation (2004–2014) periods used in this workflow.

Data is retrieved via the **polytope API**, a request-based interface to the DestinE data portal. The request specifies the CMIP6 historical activity, the ICON model, and the surface variables needed: 2 m temperature (`t2m`), total precipitation (`tp`), and surface solar radiation downwards (`ssrd`).

Because the data arrives on a HEALPix grid (a pixelised sphere, not a regular lat/lon grid), it is first regridded to a 0.5° regular grid using `earthkit.regrid` before being clipped to the catchment shapefile.

> **Note:** If `generate` is slow, this is expected — the polytope API retrieves multi-year GRIB data and the regridding adds processing time. The result is cached to disk so subsequent calls use `load`.

In [ ]:
# Construct path for DestinE historical forcing
# (not stored in settings.json to avoid changing step_0a)
path_DestinE_historical = Path(settings["base_path"]) / "forcing_data" / settings["country"] / settings["caravan_id"] / "DestinE_historical"
path_DestinE_historical.mkdir(exist_ok=True, parents=True)

# Generate forcing for the calibration + validation period
try:
    DestinE_hist_forcing_object = DestinEHistoricalForcing.load(path_DestinE_historical)
except:
    DestinE_hist_forcing_object = DestinEHistoricalForcing.generate(
        start_time=settings["calibration_start_date"],
        end_time=settings["validation_end_date"],
        directory=path_DestinE_historical,
        shape=settings["path_shape"],
    )

display(DestinE_hist_forcing_object)

In [ ]:
# Quick plot of the precipitation and potential evaporation data
plt.figure()
ds_DestinE_hist = xr.open_mfdataset([
    DestinE_hist_forcing_object["pr"],
    DestinE_hist_forcing_object["evspsblpot"]
])
ds_DestinE_hist["pr"].plot(label="precipitation")
ds_DestinE_hist["evspsblpot"].plot(label="potential evaporation")
plt.legend()
plt.title("DestinE CMIP6 historical (ICON)")